# Notes

В профиле на платформе fatsecret надо в вайтлист айпиадресов внести айпи своего приложения

# Import

In [22]:
import os
import webbrowser
from getpass import getpass
from urllib.parse import parse_qs
from dotenv import load_dotenv
from requests_oauthlib import OAuth1Session
from datetime import date
import pandas as pd

# Аутентификация

In [ ]:
load_dotenv()
CLIENT_ID = os.getenv('ConsumerKey')
CLIENT_SECRET = os.getenv('ConsumerSecret')

def parse_token(text):
    data = parse_qs(text)
    return (data.get('oauth_token', [None])[0],
            data.get('oauth_token_secret', [None])[0])

session = OAuth1Session(CLIENT_ID, CLIENT_SECRET, callback_uri='oob', signature_type='BODY')
resp = session.post('https://authentication.fatsecret.com/oauth/request_token')
resp.raise_for_status()
request_token, request_token_secret = parse_token(resp.text)

auth_url = f'https://authentication.fatsecret.com/oauth/authorize?oauth_token={request_token}'
print(f'Перейдите по ссылке и разрешите доступ:\n{auth_url}')
webbrowser.open(auth_url)

verifier = getpass('Введите PIN-код: ').strip()

access_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=request_token,
    resource_owner_secret=request_token_secret,
    verifier=verifier,
    signature_type='BODY'
)
resp = access_session.post('https://authentication.fatsecret.com/oauth/access_token')
resp.raise_for_status()
access_token, access_token_secret = parse_token(resp.text)

api_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=access_token,
    resource_owner_secret=access_token_secret
)
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'profile.get', 'format': 'json'}
)
response.raise_for_status()
print('Данные профиля:')
print(response.json())

Перейдите по ссылке и разрешите доступ:
https://authentication.fatsecret.com/oauth/authorize?oauth_token=169be8b3a56045c2b12cdfef610f06e2
Данные профиля:
{'profile': {'goal_weight_kg': '92.0000', 'height_cm': '185.00', 'height_measure': 'Cm', 'last_weight_date_int': '20507', 'last_weight_kg': '87.5000', 'weight_measure': 'Kg'}}


In [ ]:
def date_to_int(d):
    epoch = date(1970, 1, 1)
    return (d - epoch).days

start_date = date_to_int(date(2026, 4, 1))
end_date = date_to_int(date(2026, 4, 26))


response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'food_entries.get.v2', 
            'format': 'json',
            'date': start_date}
)
response.raise_for_status()

In [29]:
pd.DataFrame(response.json()['food_entries']['food_entry'])

,calories,carbohydrate,cholesterol,date_int,fat,fiber,food_entry_description,food_entry_id,food_entry_name,food_id,meal,monounsaturated_fat,number_of_units,polyunsaturated_fat,potassium,protein,saturated_fat,serving_id,sodium,sugar
0,250,55.93,0,20544,0.72,13.5,301 g Булгур (Вареный),23079810677,Булгур (Вареный),39693,Breakfast,0.093,301.000,0.295,205,9.27,0.126,62424,15,0.30
1,167,10.50,NaN,20544,5.00,NaN,2 1/2 servings Савушкин Продукт Йогурт Греческ...,23077583829,Савушкин Продукт Йогурт Греческий Teos 2%,17400944,Breakfast,NaN,2.500,NaN,NaN,20.00,NaN,16411429,NaN,19.50
2,540,91.80,NaN,20544,17.55,NaN,1.35 servings Красный Пищевик Зефир в Шоколаде,23077490604,Красный Пищевик Зефир в Шоколаде,11083037,Breakfast,NaN,1.350,NaN,NaN,4.05,NaN,10548954,NaN,NaN
3,122,27.74,NaN,20544,0,NaN,0.38 serving Красный Пищевик Мармелад,23077490603,Красный Пищевик Мармелад,78571731,Breakfast,NaN,0.380,NaN,NaN,2.85,NaN,63621977,NaN,NaN
4,617,10.89,NaN,20544,36.30,NaN,3 2/3 servings Петелинка Шницель из Куриной Гр...,23083560323,Петелинка Шницель из Куриной Грудки,81764479,Lunch,NaN,3.630,NaN,NaN,61.71,NaN,65982567,NaN,NaN
5,239,6.58,85,20544,3.19,NaN,1.879 servings Ultra Whey Pro,23083560257,Ultra Whey Pro,5686909,Lunch,NaN,1.879,NaN,282,45.28,1.880,5522529,216,5.64
6,45,4.79,NaN,20544,1.53,NaN,"1.02 servings Молоко 1,5%",23083548897,"Молоко 1,5%",17052724,Lunch,NaN,1.020,NaN,NaN,3.06,NaN,16088703,102,NaN
7,358,89.60,NaN,20544,0,NaN,1.12 servings Маркет Перекресток Мармелад со В...,23087055648,Маркет Перекресток Мармелад со Вкусом Дыни,83427867,Other,NaN,1.120,NaN,NaN,0,NaN,67252390,NaN,NaN
